In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import datasets
import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda")

In [3]:
!git clone https://github.com/garythung/trashnet.git

Cloning into 'trashnet'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 45 (delta 6), reused 1 (delta 0), pack-reused 33 (from 1)
Receiving objects: 100% (45/45), 40.64 MiB | 15.29 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [4]:
!ls trashnet/data

constants.py			     one-indexed-files-notrash_val.txt
dataset-resized.zip		     one-indexed-files.txt
one-indexed-files-notrash_test.txt   resize.py
one-indexed-files-notrash_train.txt  zero-indexed-files.txt


In [5]:
!unzip trashnet/data/dataset-resized.zip -d trashnet/data/

Archive:  trashnet/data/dataset-resized.zip
   creating: trashnet/data/dataset-resized/
  inflating: trashnet/data/dataset-resized/.DS_Store  
   creating: trashnet/data/__MACOSX/
   creating: trashnet/data/__MACOSX/dataset-resized/
  inflating: trashnet/data/__MACOSX/dataset-resized/._.DS_Store  
   creating: trashnet/data/dataset-resized/cardboard/
  inflating: trashnet/data/dataset-resized/cardboard/cardboard1.jpg  
  inflating: trashnet/data/dataset-resized/cardboard/cardboard10.jpg  
  inflating: trashnet/data/dataset-resized/cardboard/cardboard100.jpg  
  inflating: trashnet/data/dataset-resized/cardboard/cardboard101.jpg  
  inflating: trashnet/data/dataset-resized/cardboard/cardboard102.jpg  
  inflating: trashnet/data/dataset-resized/cardboard/cardboard103.jpg  
  inflating: trashnet/data/dataset-resized/cardboard/cardboard104.jpg  
  inflating: trashnet/data/dataset-resized/cardboard/cardboard105.jpg  
  inflating: trashnet/data/dataset-resized/cardboard/cardboard106.jpg  
  

In [6]:
!ls trashnet/data/dataset-resized/

cardboard  glass  metal  paper	plastic  trash


In [7]:
!ls trashnet/data/dataset-resized/cardboard | wc -l
!ls trashnet/data/dataset-resized/glass | wc -l
!ls trashnet/data/dataset-resized/metal | wc -l
!ls trashnet/data/dataset-resized/paper | wc -l
!ls trashnet/data/dataset-resized/plastic | wc -l
!ls trashnet/data/dataset-resized/trash | wc -l

403
501
410
594
482
137


In [8]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [9]:
dataset = datasets.ImageFolder(
    root="trashnet/data/dataset-resized",
    transform=transform
)

In [10]:
print(len(dataset))
print(dataset.classes)

2527
['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [11]:
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

print(train_size, val_size, test_size)

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

2021 252 254


In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [13]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [14]:
class WasteClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.dropout = nn.Dropout(p=0.5)


        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, 6)

    def forward(self, x):

      x = self.block1(x)
      x = self.block2(x)
      x = self.block3(x)

      x = self.global_avg_pool(x)
      x = x.view(x.size(0), -1)
      x = self.dropout(x)
      x = self.fc(x)
      return x

In [15]:
model = WasteClassifier().to(device)
print(model)

WasteClassifier(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (global_avg_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Linear(in_features=128, out_features=6, bias=True)
)


In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [17]:
epochs = 100

for epoch in range(epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
      images = images.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)

      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      predicted = outputs.argmax(dim=1)
      correct += (predicted == labels).sum().item()
      total += labels.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
      for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        val_correct += (predicted == labels).sum().item()
        val_total += labels.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    epoch_val_loss = val_loss / len(val_loader)
    epoch_val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f} - Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")

Epoch 1/100 - Loss: 1.6052 - Acc: 0.3137 - Val Loss: 1.4381 - Val Acc: 0.3770
Epoch 2/100 - Loss: 1.4725 - Acc: 0.4097 - Val Loss: 1.3751 - Val Acc: 0.4365
Epoch 3/100 - Loss: 1.3916 - Acc: 0.4320 - Val Loss: 1.3115 - Val Acc: 0.4802
Epoch 4/100 - Loss: 1.3155 - Acc: 0.4765 - Val Loss: 1.2435 - Val Acc: 0.4722
Epoch 5/100 - Loss: 1.2938 - Acc: 0.4735 - Val Loss: 1.2451 - Val Acc: 0.4802
Epoch 6/100 - Loss: 1.2510 - Acc: 0.4968 - Val Loss: 1.1814 - Val Acc: 0.5238
Epoch 7/100 - Loss: 1.2016 - Acc: 0.5359 - Val Loss: 1.1390 - Val Acc: 0.5357
Epoch 8/100 - Loss: 1.1922 - Acc: 0.5463 - Val Loss: 1.1562 - Val Acc: 0.5437
Epoch 9/100 - Loss: 1.1642 - Acc: 0.5423 - Val Loss: 1.1037 - Val Acc: 0.5357
Epoch 10/100 - Loss: 1.1461 - Acc: 0.5463 - Val Loss: 1.1077 - Val Acc: 0.5595
Epoch 11/100 - Loss: 1.1466 - Acc: 0.5557 - Val Loss: 1.1475 - Val Acc: 0.5317
Epoch 12/100 - Loss: 1.1242 - Acc: 0.5631 - Val Loss: 1.0808 - Val Acc: 0.5476
Epoch 13/100 - Loss: 1.1090 - Acc: 0.5705 - Val Loss: 1.0880 

In [18]:
test_loss = 0.0
test_correct = 0
test_total = 0
model.eval()

with torch.no_grad():
  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)
    test_loss += loss.item()
    predicted = outputs.argmax(dim=1)
    test_correct += (predicted == labels).sum().item()
    test_total += labels.size(0)

  print(f"Test Loss: {test_loss / len(test_loader):.4f} - Test Acc: {test_correct / test_total:.4f}")

Test Loss: 0.5148 - Test Acc: 0.8031


In [19]:
torch.save(model.state_dict(), 'waste_classifier.pth')

In [20]:
from PIL import Image

def predict(image_path, model, device):

  model.eval()

  image = Image.open(image_path).convert('RGB')
  image = transform(image)
  image = image.unsqueeze(0)
  image = image.to(device)
  with torch.no_grad():
    output = model(image)
    prediction = output.argmax(dim=1).item()

  return dataset.classes[prediction]

In [21]:
from google.colab import files

uploaded = files.upload()

Saving image2.png to image2.png


In [22]:
image_path = list(uploaded.keys())[0]

print(predict(image_path, model, device))

cardboard


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
